Langkah 3: chunking dan deduplikasi

3.1a

In [ ]:
import os, re, subprocess
from pathlib import Path

REPO = "llm-hallucination"
if not Path(REPO).exists():
    subprocess.run(["git", "clone", f"https://github.com/Tonggi-Simanjuntak/{REPO}.git"], check=True)

folder = Path(REPO) / "data" / "raw" / "imigrasi"
berkas = sorted(folder.glob("*.txt"))
print("Jumlah berkas:", len(berkas), "\n")

AWAL, AKHIR = "empty heading", "terakhir update"

for f in berkas:
    baris = f.read_text(encoding="utf-8", errors="replace").splitlines()
    idx_awal  = [i for i, b in enumerate(baris) if AWAL  in b.lower()]
    idx_akhir = [i for i, b in enumerate(baris) if AKHIR in b.lower()]
    kata = sum(len(b.split()) for b in baris)
    print(f"{f.name}")
    print(f"   baris={len(baris)} kata={kata} | penanda awal di baris {idx_awal} | penanda akhir di baris {idx_akhir}")
    print(f"   baris 1 : {baris[0][:90] if baris else '-'}")
    print(f"   baris -1: {baris[-1][:90] if baris else '-'}")
    if len(idx_awal) != 1 or len(idx_akhir) != 1:
        print("   >>> PERIKSA: penanda tidak tepat satu kali")

Jumlah berkas: 11 

ImigrasiAnakLahirDiLuarnegri.txt
   baris=96 kata=284 | penanda awal di baris [30] | penanda akhir di baris [70]
   baris 1 : https://imigrasi.go.id/layanan-wni/paspor-republik-indonesia/permohonan-baru/anak-yang-lah
   baris -1: 
ImigrasiHajiUmroh.txt
   baris=111 kata=416 | penanda awal di baris [30] | penanda akhir di baris [85]
   baris 1 : https://imigrasi.go.id/layanan-wni/paspor-republik-indonesia/permohonan-baru/haji-umroh
   baris -1: 
ImigrasiPasporRusak.txt
   baris=151 kata=582 | penanda awal di baris [30] | penanda akhir di baris [125]
   baris 1 : https://imigrasi.go.id/layanan-wni/paspor-republik-indonesia/penggantian/paspor-rusak
   baris -1: 
ImigrasiPengubahandataPaspor.txt
   baris=100 kata=309 | penanda awal di baris [30] | penanda akhir di baris [74]
   baris 1 : https://imigrasi.go.id/layanan-wni/paspor-republik-indonesia/penggantian/pengubahan-data-p
   baris -1: 
imigrasiAnakDwikewarganegaraan.txt
   baris=135 kata=675 | penanda awal di baris

3.1b

In [ ]:
import subprocess, sys
from pathlib import Path
import pandas as pd

REPO = "llm-hallucination"
AWAL, AKHIR = "empty heading", "terakhir update"

# 1) Pastikan salinan repo ada dan terbaru
if not Path(REPO).exists():
    subprocess.run(["git", "clone", f"https://github.com/Tonggi-Simanjuntak/{REPO}.git"], check=True)
else:
    out = subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True)
    print("git pull:", (out.stdout + out.stderr).strip(), "\n")

folder = Path(REPO) / "data" / "raw" / "imigrasi"
berkas = sorted(folder.glob("*.txt"))
print("Jumlah berkas:", len(berkas), "\n")

keluar = Path(REPO) / "data" / "processed" / "konten_inti"
keluar.mkdir(parents=True, exist_ok=True)

# 2) Ekstrak konten inti (di antara dua penanda, penanda tidak ikut)
rows, contoh = [], {}
for f in berkas:
    baris = f.read_text(encoding="utf-8", errors="replace").splitlines()
    idx0 = [i for i, b in enumerate(baris) if AWAL in b.lower()]
    idx1 = [i for i, b in enumerate(baris) if AKHIR in b.lower()]
    if len(idx0) != 1 or len(idx1) != 1:
        print(f"PERINGATAN: {f.name} dilewati (penanda awal={idx0}, akhir={idx1})")
        continue
    i0, i1 = idx0[0], idx1[0]

    isi = [b.strip() for b in baris[i0 + 1:i1]]
    bersih = []                       # buang baris kosong berurutan
    for b in isi:
        if b or (bersih and bersih[-1]):
            bersih.append(b)
    teks = "\n".join(bersih).strip()

    (keluar / f.name).write_text(teks, encoding="utf-8")
    contoh[f.name] = teks
    rows.append(dict(
        berkas=f.name,
        url=baris[0].strip() if baris and baris[0].startswith("http") else "",
        tanggal_pembaruan=baris[i1].strip(),
        kata_total=sum(len(b.split()) for b in baris),
        kata_inti=len(teks.split()),
    ))

# 3) Ringkasan
df = pd.DataFrame(rows)
df["persen_inti"] = (df.kata_inti / df.kata_total * 100).round(1)
print(df.drop(columns="url").to_string(index=False))
print("\nTOTAL kata:", df.kata_total.sum(), "| inti:", df.kata_inti.sum(),
      "| persen inti:", round(df.kata_inti.sum() / df.kata_total.sum() * 100, 1))
print("Berkas tanpa URL di baris 1:", df[df.url == ""].berkas.tolist())
print("Kelompok tanggal pembaruan:\n", df.tanggal_pembaruan.value_counts().to_string())
df.to_csv(Path(REPO) / "data" / "processed" / "ringkasan_konten_inti.csv", index=False)

# 4) Contoh isi (dua berkas)
for nama in list(contoh)[:2]:
    print(f"\n--- Contoh isi: {nama} ---\n{contoh[nama][:500]}")

git pull: Already up to date. 

Jumlah berkas: 11 

                                  berkas             tanggal_pembaruan  kata_total  kata_inti  persen_inti
        ImigrasiAnakLahirDiLuarnegri.txt Terakhir Update : 11 Jan 2024         284        177         62.3
                   ImigrasiHajiUmroh.txt Terakhir Update : 11 Jan 2024         416        309         74.3
                 ImigrasiPasporRusak.txt  Terakhir Update : 2 Sep 2025         582        475         81.6
        ImigrasiPengubahandataPaspor.txt Terakhir Update : 11 Jan 2024         309        202         65.4
      imigrasiAnakDwikewarganegaraan.txt Terakhir Update : 11 Jan 2024         675        569         84.3
               imigrasiAnakdiBawah17.txt Terakhir Update : 11 Jan 2024         370        264         71.4
          imigrasiCalonPekerjaMigran.txt Terakhir Update : 11 Jan 2024         474        367         77.4
              imigrasiMasyarakatUmum.txt  Terakhir Update : 2 Sep 2025         459        35

3.1c

In [ ]:
import subprocess
from pathlib import Path

REPO = "llm-hallucination"
AWAL, AKHIR = "empty heading", "terakhir update"
PENANDA_ISI = "informasi umum"

if not Path(REPO).exists():
    subprocess.run(["git", "clone", f"https://github.com/Tonggi-Simanjuntak/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True)

berkas = sorted((Path(REPO) / "data" / "raw" / "imigrasi").glob("*.txt"))

menu_per_berkas = {}
for f in berkas:
    baris = f.read_text(encoding="utf-8", errors="replace").splitlines()
    i0 = next(i for i, b in enumerate(baris) if AWAL in b.lower())
    i1 = next(i for i, b in enumerate(baris) if AKHIR in b.lower())
    inti = [b.strip() for b in baris[i0 + 1:i1]]
    inti = [b for b in inti if b]                      # abaikan baris kosong

    idx = [i for i, b in enumerate(inti) if b.lower() == PENANDA_ISI]
    status = "OK" if len(idx) == 2 else "PERIKSA"
    print(f"{f.name}")
    print(f"   'Informasi Umum' berdiri sendiri di baris (inti) {idx} -> {status}")

    if len(idx) >= 2:
        menu = inti[:idx[1]]                           # semua di atas kemunculan kedua
        isi = inti[idx[1]:]
        menu_per_berkas[f.name] = menu
        print(f"   menu: {len(menu)} baris, {sum(len(b.split()) for b in menu)} kata"
              f" | isi layanan: {sum(len(b.split()) for b in isi)} kata")

# Bandingkan menu antar-berkas
if menu_per_berkas:
    semua = [set(m) for m in menu_per_berkas.values()]
    umum = set.intersection(*semua)
    gabungan = set.union(*semua)
    print("\n--- Baris menu yang ada di SEMUA berkas:", len(umum))
    print("--- Baris menu yang tidak sama di semua berkas:", sorted(gabungan - umum))

    nama0 = next(iter(menu_per_berkas))
    print(f"\n--- Blok menu lengkap dari {nama0} ---")
    for b in menu_per_berkas[nama0]:
        print("  |", b)

ImigrasiAnakLahirDiLuarnegri.txt
   'Informasi Umum' berdiri sendiri di baris (inti) [9, 14] -> OK
   menu: 14 baris, 33 kata | isi layanan: 144 kata
ImigrasiHajiUmroh.txt
   'Informasi Umum' berdiri sendiri di baris (inti) [9, 14] -> OK
   menu: 14 baris, 33 kata | isi layanan: 276 kata
ImigrasiPasporRusak.txt
   'Informasi Umum' berdiri sendiri di baris (inti) [8, 13] -> OK
   menu: 13 baris, 29 kata | isi layanan: 446 kata
ImigrasiPengubahandataPaspor.txt
   'Informasi Umum' berdiri sendiri di baris (inti) [12, 34] -> OK
   menu: 34 baris, 176 kata | isi layanan: 26 kata
imigrasiAnakDwikewarganegaraan.txt
   'Informasi Umum' berdiri sendiri di baris (inti) [9, 14] -> OK
   menu: 14 baris, 33 kata | isi layanan: 536 kata
imigrasiAnakdiBawah17.txt
   'Informasi Umum' berdiri sendiri di baris (inti) [9, 14] -> OK
   menu: 14 baris, 33 kata | isi layanan: 231 kata
imigrasiCalonPekerjaMigran.txt
   'Informasi Umum' berdiri sendiri di baris (inti) [9, 14] -> OK
   menu: 14 baris, 33 kata 

3.1d

In [ ]:
import subprocess
from pathlib import Path

REPO = "llm-hallucination"
AWAL, AKHIR = "empty heading", "terakhir update"

if not Path(REPO).exists():
    subprocess.run(["git", "clone", f"https://github.com/Tonggi-Simanjuntak/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True)

f = Path(REPO) / "data" / "raw" / "imigrasi" / "ImigrasiPengubahandataPaspor.txt"
baris = f.read_text(encoding="utf-8", errors="replace").splitlines()
i0 = next(i for i, b in enumerate(baris) if AWAL in b.lower())
i1 = next(i for i, b in enumerate(baris) if AKHIR in b.lower())
inti = [b.strip() for b in baris[i0 + 1:i1] if b.strip()]

print(f"{f.name}: {len(inti)} baris inti\n")
for n, b in enumerate(inti):
    tanda = "  <<< Informasi Umum" if b.lower() == "informasi umum" else ""
    print(f"{n:3d} | {b[:110]}{tanda}")

ImigrasiPengubahandataPaspor.txt: 36 baris inti

  0 | Paspor Republik Indonesia
  1 | PERMOHONAN BARU
  2 | PENGGANTIAN
  3 | Paspor Akan Habis Masa Berlaku
  4 | Paspor Rusak
  5 | Paspor Hilang
  6 | Penggantian Paspor Diluar Negeri
  7 | Pengubahan Data Paspor
  8 | Biaya
  9 | Mekanisme Pengesahan
 10 | Prosedur
 11 | Persyaratan
 12 | Informasi Umum  <<< Informasi Umum
 13 | Biaya
 14 | Pelayanan ini dikenakan biaya penggantian paspor.
 15 | Mekanisme Pengesahan
 16 | Serahkan berkas persyaratan (asli dan fotokopi) serta Perdim 11 (terdapat di kantor imigrasi) yang telah diisi
 17 | Petugas akan memproses perubahan data paspor.
 18 | Pejabat Imigrasi akan menyetujui perubahan data paspor.
 19 | Petugas akan mencetak paspor baru Anda setelah mendapatkan persetujuan Direktur Jenderal Imigrasi.
 20 | Paspor baru yang telah selesai akan diberikan kepada Anda.
 21 | Jika terjadi penolakan dalam perubahan data paspor Anda, penyebabnya adalah sebagai berikut.
 22 | Persyaratan tidak len

3.1e

In [ ]:
import subprocess
from pathlib import Path
import pandas as pd

REPO = "llm-hallucination"
AWAL, AKHIR = "empty heading", "terakhir update"
LABEL = {"informasi umum", "persyaratan", "prosedur", "biaya",
         "mekanisme pengesahan", "mekanisme penerbitan"}

if not Path(REPO).exists():
    subprocess.run(["git", "clone", f"https://github.com/Tonggi-Simanjuntak/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True)

berkas = sorted((Path(REPO) / "data" / "raw" / "imigrasi").glob("*.txt"))
keluar = Path(REPO) / "data" / "processed" / "konten_inti"
keluar.mkdir(parents=True, exist_ok=True)

rows, contoh = [], {}
for f in berkas:
    baris = f.read_text(encoding="utf-8", errors="replace").splitlines()
    idx0 = [i for i, b in enumerate(baris) if AWAL in b.lower()]
    idx1 = [i for i, b in enumerate(baris) if AKHIR in b.lower()]
    if len(idx0) != 1 or len(idx1) != 1:
        print(f"PERIKSA: {f.name} penanda awal/akhir tidak tepat satu kali")
        continue
    inti = [b.strip() for b in baris[idx0[0] + 1:idx1[0]] if b.strip()]

    # menu berakhir sebelum nama tab pertama kali berulang
    lihat, akhir_menu = set(), None
    for i, b in enumerate(inti):
        k = b.lower()
        if k in LABEL:
            if k in lihat:
                akhir_menu = i
                break
            lihat.add(k)

    if akhir_menu is None:
        print(f"PERIKSA: {f.name} tidak ditemukan batas menu")
        continue

    menu, isi = inti[:akhir_menu], inti[akhir_menu:]
    urutan = [b for b in isi if b.lower() in LABEL]
    if not isi or isi[0].lower() not in LABEL:
        print(f"PERIKSA: {f.name} isi tidak diawali nama bagian: {isi[:1]}")

    (keluar / f.name).write_text("\n".join(isi), encoding="utf-8")
    contoh[f.name] = isi
    rows.append(dict(
        berkas=f.name,
        url=baris[0].strip() if baris and baris[0].startswith("http") else "",
        tanggal_pembaruan=baris[idx1[0]].strip(),
        kata_total=sum(len(b.split()) for b in baris),
        kata_menu=sum(len(b.split()) for b in menu),
        kata_isi=sum(len(b.split()) for b in isi),
        urutan_bagian=" > ".join(urutan),
    ))

df = pd.DataFrame(rows)
print(df.drop(columns=["url", "urutan_bagian"]).to_string(index=False))
t_total, t_isi = df.kata_total.sum(), df.kata_isi.sum()
print(f"\nTOTAL kata: {t_total} | menu dibuang: {df.kata_menu.sum()} | isi layanan: {t_isi}"
      f" | persen isi: {round(t_isi / t_total * 100, 1)}")
print("Perkiraan manual: isi sekitar 3486 kata")

print("\n--- Urutan sub-bagian di dalam isi ---")
for _, r in df.iterrows():
    print(f"{r.berkas}: {r.urutan_bagian}")

df.to_csv(Path(REPO) / "data" / "processed" / "ringkasan_konten_inti.csv", index=False)

nama = "ImigrasiPengubahandataPaspor.txt"
print(f"\n--- 6 baris pertama isi: {nama} ---")
for b in contoh.get(nama, [])[:6]:
    print("|", b[:100])

                                  berkas             tanggal_pembaruan  kata_total  kata_menu  kata_isi
        ImigrasiAnakLahirDiLuarnegri.txt Terakhir Update : 11 Jan 2024         284         33       144
                   ImigrasiHajiUmroh.txt Terakhir Update : 11 Jan 2024         416         33       276
                 ImigrasiPasporRusak.txt  Terakhir Update : 2 Sep 2025         582         29       446
        ImigrasiPengubahandataPaspor.txt Terakhir Update : 11 Jan 2024         309         29       173
      imigrasiAnakDwikewarganegaraan.txt Terakhir Update : 11 Jan 2024         675         33       536
               imigrasiAnakdiBawah17.txt Terakhir Update : 11 Jan 2024         370         33       231
          imigrasiCalonPekerjaMigran.txt Terakhir Update : 11 Jan 2024         474         33       334
              imigrasiMasyarakatUmum.txt  Terakhir Update : 2 Sep 2025         459         33       318
  imigrasiPasporAkanHabisMasaBerlaku.txt  Terakhir Update : 2 Se

Selanjutnya proses chunking pdf Duckcapil

In [ ]:
import subprocess, sys, re
from pathlib import Path
from collections import Counter
import pandas as pd

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf"], check=True)
from pypdf import PdfReader

REPO = "llm-hallucination"

# 1) Pastikan salinan repo ada dan terbaru
if not Path(REPO).exists():
    subprocess.run(["git", "clone", f"https://github.com/Tonggi-Simanjuntak/{REPO}.git"], check=True)
else:
    out = subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True)
    print("git pull:", (out.stdout + out.stderr).strip(), "\n")

# 2) Cari PDF (tidak peduli huruf besar-kecil)
daftar = sorted(p for p in Path(REPO).rglob("*") if p.suffix.lower() == ".pdf")
print("PDF ditemukan:", [str(p) for p in daftar])
assert daftar, "Tidak ada PDF di salinan repo ini"
pdf = daftar[0]

# 3) Ekstrak teks per halaman
r = PdfReader(str(pdf))
n_hal = len(r.pages)
print(f"\nDipakai: {pdf.name} | {n_hal} halaman\n")

keluar = Path(REPO) / "data" / "processed" / "dukcapil_teks"
keluar.mkdir(parents=True, exist_ok=True)

halaman, rows = [], []
for i, hal in enumerate(r.pages, start=1):
    t = (hal.extract_text() or "").strip()
    halaman.append(t)
    (keluar / f"dukcapil_p{i:02d}.txt").write_text(t, encoding="utf-8")
    rows.append(dict(halaman=i, karakter=len(t), kata=len(t.split())))

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\nTOTAL kata:", df.kata.sum(),
      "| halaman hampir kosong (<200 karakter):", df[df.karakter < 200].halaman.tolist())
df.to_csv(Path(REPO) / "data" / "processed" / "ringkasan_dukcapil.csv", index=False)

# 4) Cari baris yang berulang di banyak halaman (calon header/footer)
hitung = Counter()
for t in halaman:
    for b in set(x.strip() for x in t.splitlines() if x.strip()):
        hitung[b] += 1
batas = max(3, int(0.4 * n_hal))
print(f"\n--- Baris yang muncul di >= {batas} halaman (calon header/footer) ---")
for b, n in hitung.most_common(15):
    if n >= batas:
        print(f"{n:3d}x | {b[:100]}")

# 5) Contoh teks
for nomor in (1, 5):
    if n_hal >= nomor:
        print(f"\n--- Contoh teks halaman {nomor} ---\n", halaman[nomor - 1][:800])

git pull: Already up to date. 

PDF ditemukan: ['llm-hallucination/data/raw/dukcapil/dukcapil_syarat_prosedur.pdf']

Dipakai: dukcapil_syarat_prosedur.pdf | 23 halaman

 halaman  karakter  kata
       1        89    13
       2      1642   227
       3       420    59
       4      1481   199
       5       632    90
       6      1405   193
       7       491    69
       8       316    46
       9       927   128
      10       448    66
      11       943   131
      12      1793   276
      13      1317   204
      14      2401   344
      15      1363   197
      16      1115   172
      17       549    83
      18      1003   153
      19       417    64
      20      1102   161
      21       735   105
      22       670    99
      23        64     8

TOTAL kata: 3087 | halaman hampir kosong (<200 karakter): [1, 23]

--- Baris yang muncul di >= 9 halaman (calon header/footer) ---
 17x | PENJELASAN:
 10x | Harus memenuhi persyaratan:

--- Contoh teks halaman 1 ---
 SYARAT DAN PE

In [ ]:
import re

print("--- Halaman 23 ---\n", halaman[22])

print("\n--- Baris judul (huruf kapital semua) per halaman ---")
for i, t in enumerate(halaman, start=1):
    for b in t.splitlines():
        s = b.strip()
        if len(s) >= 12 and s == s.upper() and re.search(r"[A-Z]", s):
            print(f"hal {i:2d}: {s[:100]}")

--- Halaman 23 ---
 TERIMA KASIH
DITJEN KEPENDUDUKAN DAN PENCATATAN SIPIL
KEMENDAGRI

--- Baris judul (huruf kapital semua) per halaman ---
hal  1: SYARAT DAN PENJELASAN DALAM
hal  1: LAYANAN PINDAH DATANG PENDUDUK,
hal  1: BIODATA, KK, KTP-EL DAN KIA
hal  2: PENERBITAN BIODATA WNI
hal  3: PENERBITAN BIODATA OA
hal  4: PENERBITAN KARTU KELUARGA BARU
hal  4: KARENA MEMBENTUK KELUARGA BARU
hal  4: PENERBITAN KARTU KELUARGA BARU
hal  4: KARENA PENGGANTIAN KEPALA KELUARGA
hal  5: PENERBITAN KARTU KELUARGA BARU
hal  5: KARENA PISAH KK DALAM 1 (SATU) ALAMAT
hal  6: PENERBITAN KARTU KELUARGA KARENA PERUBAHAN DATA
hal  7: PENERBITAN KARTU KELUARGA KARENA HILANG/RUSAK
hal  8: PENERBITAN KTP-EL BARU UNTUK WNI
hal  9: PENERBITAN KTP-EL BARU KARENA PINDAH, PERUBAHAN DATA,
hal  9: RUSAK DAN HILANG UNTUK WNI
hal 10: PENERBITAN KTP-EL BARU UNTUK OA
hal 11: PENERBITAN KTP-EL BARU KARENA PINDAH, PERUBAHAN DATA,
hal 11: RUSAK, HILANG DAN PERPANJANGAN UNTUK OA
hal 12: PENERBITAN KARTU IDENTITAS ANAK BARU

3.4a

In [ ]:
import subprocess, re, statistics
from pathlib import Path
from collections import defaultdict, Counter

REPO = "llm-hallucination"
AWAL, AKHIR = "empty heading", "terakhir update"
LABEL = {"informasi umum", "persyaratan", "prosedur", "biaya",
         "mekanisme pengesahan", "mekanisme penerbitan"}

if not Path(REPO).exists():
    subprocess.run(["git", "clone", f"https://github.com/Tonggi-Simanjuntak/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True)

def isi_layanan(path):
    baris = path.read_text(encoding="utf-8", errors="replace").splitlines()
    i0 = next(i for i, b in enumerate(baris) if AWAL in b.lower())
    i1 = next(i for i, b in enumerate(baris) if AKHIR in b.lower())
    inti = [b.strip() for b in baris[i0 + 1:i1] if b.strip()]
    lihat, akhir = set(), None
    for i, b in enumerate(inti):
        k = b.lower()
        if k in LABEL:
            if k in lihat:
                akhir = i
                break
            lihat.add(k)
    return inti[akhir:]

def bagi(isi):
    out, label, buf = [], None, []
    for b in isi:
        if b.lower() in LABEL:
            if label is not None:
                out.append((label, buf))
            label, buf = b, []
        else:
            buf.append(b)
    if label is not None:
        out.append((label, buf))
    return out

def norm(s):
    return re.sub(r"\s+", " ", s.lower()).strip()

berkas = sorted((Path(REPO) / "data" / "raw" / "imigrasi").glob("*.txt"))
data = {f.name: bagi(isi_layanan(f)) for f in berkas}
print("Jumlah halaman:", len(data), "| total sub-bagian:", sum(len(v) for v in data.values()), "\n")

# ---------- A. Tingkat baris (>= 40 karakter) ----------
hitung, di_halaman = Counter(), defaultdict(set)
for nama, bagian in data.items():
    for label, buf in bagian:
        for b in buf:
            if len(b) >= 40:
                k = norm(b)
                hitung[k] += 1
                di_halaman[k].add(nama)
total = sum(hitung.values())
unik = len(hitung)
banyak = sum(1 for s in di_halaman.values() if len(s) >= 2)
print("=== Tingkat baris (>= 40 karakter) ===")
print(f"kejadian baris   : {total}")
print(f"baris unik       : {unik}")
print(f"unik muncul di >=2 halaman: {banyak} ({banyak / unik * 100:.0f}% dari baris unik)")
print(f"salinan berlebih : {total - unik} ({(total - unik) / total * 100:.0f}% dari kejadian)\n")

# ---------- B. Tingkat sub-bagian ----------
kelompok = defaultdict(list)
panjang, kosong = [], 0
for nama, bagian in data.items():
    for label, buf in bagian:
        teks = " ".join(buf)
        kata = len(teks.split())
        if kata == 0:
            kosong += 1
            continue
        panjang.append(kata)
        kelompok[norm(teks)].append((nama, label, kata))

n_sub = len(panjang)
ganda = {k: v for k, v in kelompok.items() if len(v) > 1}
print("=== Tingkat sub-bagian ===")
print(f"sub-bagian berisi: {n_sub} (kosong: {kosong})")
print(f"teks unik        : {len(kelompok)}")
print(f"kelompok identik (>=2 halaman): {len(ganda)} | sub-bagian di dalamnya: "
      f"{sum(len(v) for v in ganda.values())}\n")

for k, v in sorted(ganda.items(), key=lambda x: -len(x[1]))[:12]:
    print(f"{len(v)}x | {v[0][1]} | {v[0][2]} kata | {k[:80]}")
    print("     di:", ", ".join(n.replace('.txt', '') for n, _, _ in v))

print("\n=== Panjang sub-bagian (kata) ===")
print(f"min {min(panjang)} | median {statistics.median(panjang)} | max {max(panjang)}")
print(f"< 20 kata: {sum(p < 20 for p in panjang)} | < 80 kata: {sum(p < 80 for p in panjang)}"
      f" | > 250 kata: {sum(p > 250 for p in panjang)}")

Jumlah halaman: 11 | total sub-bagian: 55 

=== Tingkat baris (>= 40 karakter) ===
kejadian baris   : 243
baris unik       : 137
unik muncul di >=2 halaman: 50 (36% dari baris unik)
salinan berlebih : 106 (44% dari kejadian)

=== Tingkat sub-bagian ===
sub-bagian berisi: 55 (kosong: 0)
teks unik        : 41
kelompok identik (>=2 halaman): 9 | sub-bagian di dalamnya: 23

4x | Biaya | 28 kata | paspor biasa nonelektronik 48 halaman: rp350.000 paspor biasa elektronik 48 hala
     di: ImigrasiHajiUmroh, imigrasiAnakDwikewarganegaraan, imigrasiAnakdiBawah17, imigrasiCalonPekerjaMigran
4x | Mekanisme Penerbitan | 16 kata | pemeriksaan kelengkapan dan keabsahan persyaratan pembayaran biaya paspor pengam
     di: imigrasiAnakDwikewarganegaraan, imigrasiAnakdiBawah17, imigrasiCalonPekerjaMigran, imigrasiMasyarakatUmum
3x | Biaya | 81 kata | 1 layanan percepatan paspor selesai pada hari yang sama rp 1.000.000,- 2 paspor 
     di: imigrasiMasyarakatUmum, imigrasiPasporAkanHabisMasaBerlaku, imigra

3.2b

In [ ]:
import subprocess, sys, re, statistics
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf"], check=True)
from pypdf import PdfReader

REPO = "llm-hallucination"
if not Path(REPO).exists():
    subprocess.run(["git", "clone", f"https://github.com/Tonggi-Simanjuntak/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True)

pdf = sorted(p for p in Path(REPO).rglob("*") if p.suffix.lower() == ".pdf")[0]
r = PdfReader(str(pdf))
n_hal = len(r.pages)
print(f"{pdf.name}: {n_hal} halaman | dipakai: halaman 2 sampai {n_hal - 1}\n")

def norm(s):
    return re.sub(r"\s+", " ", s.lower()).strip()

def is_judul(s):
    return (len(s) >= 12 and s == s.upper() and re.search(r"[A-Z]", s)
            and not re.match(r"^PENJELASAN\s*:?$", s))

def rapikan(teks):
    mentah = [b.strip() for b in teks.splitlines() if b.strip()]
    hasil = []
    for b in mentah:
        b = re.sub(r"^PENJELASAN\s*:\s*$", "PENJELASAN:", b)
        lanjut = (hasil and re.match(r"^[a-z]", b)
                  and not re.match(r"^[a-z][\.\)]\s", b)
                  and not is_judul(hasil[-1]))
        if lanjut:
            hasil[-1] = hasil[-1] + " " + b      # gabung baris yang patah
        else:
            hasil.append(b)
    return mentah, hasil

bagian, baris_mentah, baris_rapi = [], 0, 0
for nomor in range(2, n_hal):                      # halaman 2 .. n_hal-1 (1-based)
    mentah, baris = rapikan(r.pages[nomor - 1].extract_text() or "")
    baris_mentah += len(mentah)
    baris_rapi += len(baris)
    i = 0
    while i < len(baris):
        b = baris[i]
        if is_judul(b):
            j, bagian_judul = i, [b]
            while j + 1 < len(baris) and is_judul(baris[j + 1]):
                j += 1
                bagian_judul.append(baris[j])
            t = " ".join(bagian_judul)
            if bagian and norm(t) == norm(bagian[-1]["judul"]):
                bagian[-1]["hal_akhir"] = nomor     # lanjutan layanan yang sama
            else:
                bagian.append(dict(judul=t, hal_awal=nomor, hal_akhir=nomor, baris=[]))
            i = j + 1
        else:
            if not bagian:
                bagian.append(dict(judul="(tanpa judul)", hal_awal=nomor, hal_akhir=nomor, baris=[]))
            bagian[-1]["baris"].append(b)
            bagian[-1]["hal_akhir"] = nomor
            i += 1

print(f"Baris sebelum digabung: {baris_mentah} | sesudah: {baris_rapi}")
print(f"Jumlah bagian layanan : {len(bagian)}\n")

rows = []
for n, b in enumerate(bagian, start=1):
    teks = "\n".join(b["baris"])
    rows.append(dict(no=n, hal=f'{b["hal_awal"]}-{b["hal_akhir"]}' if b["hal_akhir"] != b["hal_awal"] else str(b["hal_awal"]),
                     kata=len(teks.split()), judul=b["judul"], teks=teks))
df = pd.DataFrame(rows)
for _, x in df.iterrows():
    print(f"{x.no:2d} | hal {x.hal:5s} | {x.kata:4d} kata | {x.judul[:70]}")

kata = df.kata.tolist()
print(f"\nTotal kata (hal 2-{n_hal - 1}): {sum(kata)}")
print(f"Panjang bagian: min {min(kata)} | median {statistics.median(kata)} | max {max(kata)}")
print(f"< 20 kata: {sum(k < 20 for k in kata)} | < 80 kata: {sum(k < 80 for k in kata)} | > 250 kata: {sum(k > 250 for k in kata)}")

# duplikasi tingkat baris (>= 40 karakter) dan tingkat bagian
hitung, di_bagian = Counter(), defaultdict(set)
for n, b in enumerate(bagian, start=1):
    for l in b["baris"]:
        if len(l) >= 40:
            k = norm(l); hitung[k] += 1; di_bagian[k].add(n)
total, unik = sum(hitung.values()), len(hitung)
banyak = sum(1 for s in di_bagian.values() if len(s) >= 2)
print(f"\nBaris >=40 karakter: kejadian {total} | unik {unik} | unik di >=2 bagian {banyak} | salinan berlebih {total - unik}")
ganda = [v for v in df.groupby(df.teks.map(norm)).no.apply(list) if len(v) > 1]
print("Bagian dengan teks identik:", ganda if ganda else "tidak ada")

df.to_csv(Path(REPO) / "data" / "processed" / "dukcapil_bagian.csv", index=False)

print("\n--- Contoh teks bersih: bagian pada halaman 5 ---")
contoh = df[df.hal == "5"]
print(contoh.iloc[0].teks[:700] if len(contoh) else "(tidak ada)")

dukcapil_syarat_prosedur.pdf: 23 halaman | dipakai: halaman 2 sampai 22

Baris sebelum digabung: 550 | sesudah: 421
Jumlah bagian layanan : 19

 1 | hal 2     |  224 kata | PENERBITAN BIODATA WNI
 2 | hal 3     |   56 kata | PENERBITAN BIODATA OA
 3 | hal 4     |  108 kata | PENERBITAN KARTU KELUARGA BARU KARENA MEMBENTUK KELUARGA BARU
 4 | hal 4     |   75 kata | PENERBITAN KARTU KELUARGA BARU KARENA PENGGANTIAN KEPALA KELUARGA
 5 | hal 5     |   79 kata | PENERBITAN KARTU KELUARGA BARU KARENA PISAH KK DALAM 1 (SATU) ALAMAT
 6 | hal 6     |  187 kata | PENERBITAN KARTU KELUARGA KARENA PERUBAHAN DATA
 7 | hal 7     |   64 kata | PENERBITAN KARTU KELUARGA KARENA HILANG/RUSAK
 8 | hal 8     |   41 kata | PENERBITAN KTP-EL BARU UNTUK WNI
 9 | hal 9     |  116 kata | PENERBITAN KTP-EL BARU KARENA PINDAH, PERUBAHAN DATA, RUSAK DAN HILANG
10 | hal 10    |   61 kata | PENERBITAN KTP-EL BARU UNTUK OA
11 | hal 11    |  118 kata | PENERBITAN KTP-EL BARU KARENA PINDAH, PERUBAHAN DATA, RUSAK, HILA

3.4b

In [ ]:
import subprocess, sys, re
from pathlib import Path
import pandas as pd

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf"], check=True)
from pypdf import PdfReader

REPO = "llm-hallucination"
if not Path(REPO).exists():
    subprocess.run(["git", "clone", f"https://github.com/Tonggi-Simanjuntak/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True)

AWAL, AKHIR = "empty heading", "terakhir update"
LABEL = {"informasi umum", "persyaratan", "prosedur", "biaya",
         "mekanisme pengesahan", "mekanisme penerbitan"}
DUKCAPIL_URL = ("https://dukcapil.kemendagri.go.id/files/2023/"
                "1680205930Syarat%20dan%20Prosedur%20Layanan%20Pendaftaran%20Penduduk.pdf")
TGL_AKSES_DUKCAPIL = "22/11/2026"

LAYANAN = {   # nama berkas (huruf kecil) -> (layanan, kelompok)
    "imigrasianaklahirdiluarnegri": ("Anak Yang Lahir Di Luar Negeri", "Permohonan Baru"),
    "imigrasihajiumroh": ("Haji Umroh", "Permohonan Baru"),
    "imigrasianakdwikewarganegaraan": ("Anak Dwikewarganegaraan", "Permohonan Baru"),
    "imigrasianakdibawah17": ("Anak Dibawah 17 Tahun", "Permohonan Baru"),
    "imigrasicalonpekerjamigran": ("Calon Pekerja Migran Indonesia", "Permohonan Baru"),
    "imigrasimasyarakatumum": ("Masyarakat Umum", "Permohonan Baru"),
    "imigrasipasporrusak": ("Paspor Rusak", "Penggantian"),
    "imigrasipengubahandatapaspor": ("Pengubahan Data Paspor", "Penggantian"),
    "imigrasipasporakanhabismasaberlaku": ("Paspor Akan Habis Masa Berlaku", "Penggantian"),
    "imigrasipasporhilang": ("Paspor Hilang", "Penggantian"),
    "imigrasipenggantianpaspordiluarnegri": ("Penggantian Paspor Di Luar Negeri", "Penggantian"),
}

def norm(s): return re.sub(r"\s+", " ", s.lower()).strip()
def slug(s): return re.sub(r"[^A-Za-z0-9]", "", s.title())
def jml_kata(x): return len(x.split())

# ================= IMIGRASI =================
def baca(path):
    baris = path.read_text(encoding="utf-8", errors="replace").splitlines()
    i0 = next(i for i, b in enumerate(baris) if AWAL in b.lower())
    i1 = next(i for i, b in enumerate(baris) if AKHIR in b.lower())
    inti = [b.strip() for b in baris[i0 + 1:i1] if b.strip()]
    lihat, akhir = set(), None
    for i, b in enumerate(inti):
        k = b.lower()
        if k in LABEL:
            if k in lihat:
                akhir = i
                break
            lihat.add(k)
    url = baris[0].strip() if baris and baris[0].startswith("http") else ""
    tgl = baris[i1].split(":", 1)[-1].strip()
    return inti[akhir:], url, tgl

def bagi(isi):
    out, label, buf = [], None, []
    for b in isi:
        if b.lower() in LABEL:
            if label is not None:
                out.append((label, buf))
            label, buf = b, []
        else:
            buf.append(b)
    if label is not None:
        out.append((label, buf))
    return out

semua = []
for f in sorted((Path(REPO) / "data" / "raw" / "imigrasi").glob("*.txt")):
    isi, url, tgl = baca(f)
    nama, kel = LAYANAN[f.stem.lower()]   # error jika berkas belum terdaftar
    for label, buf in bagi(isi):
        semua.append(dict(stem=f.stem, layanan=nama, kelompok=kel, bagian=label,
                          teks="\n".join(buf), url=url, tgl=tgl))

grup = {}
for s in semua:
    grup.setdefault((s["bagian"].lower(), norm(s["teks"])), []).append(s)

def uniq(xs): return list(dict.fromkeys(x for x in xs if x))

rows = []
for daftar in grup.values():
    a = daftar[0]
    layanan = "; ".join(uniq(d["layanan"] for d in daftar))
    kel = "; ".join(uniq(d["kelompok"] for d in daftar))
    rows.append(dict(
        chunk_id=f"imi_{a['stem'].replace('imigrasi','').replace('Imigrasi','')}_{slug(a['bagian'])}",
        sumber="imigrasi", layanan=layanan, bagian=a["bagian"],
        sumber_url="; ".join(uniq(d["url"] for d in daftar)),
        tanggal_akses="",
        tanggal_pembaruan_halaman="; ".join(uniq(d["tgl"] for d in daftar)),
        teks=a["teks"],
        teks_konteks=f"Layanan paspor RI ({kel}): {layanan}. Bagian: {a['bagian']}.\n{a['teks']}",
        sumber_ganda="; ".join(d["stem"] for d in daftar) if len(daftar) > 1 else "",
        sasaran="WNI"))
n_imi_sebelum, n_imi_sesudah = len(semua), len(rows)

# ================= DUKCAPIL =================
pdf = sorted(p for p in Path(REPO).rglob("*") if p.suffix.lower() == ".pdf")[0]
r = PdfReader(str(pdf)); n_hal = len(r.pages)

def is_judul(s):
    return (len(s) >= 12 and s == s.upper() and re.search(r"[A-Z]", s)
            and not re.match(r"^PENJELASAN\s*:?$", s))

def rapikan(teks):
    hasil = []
    for b in [x.strip() for x in teks.splitlines() if x.strip()]:
        b = re.sub(r"^PENJELASAN\s*:\s*$", "PENJELASAN:", b)
        if (hasil and re.match(r"^[a-z]", b) and not re.match(r"^[a-z][\.\)]\s", b)
                and not is_judul(hasil[-1])):
            hasil[-1] += " " + b
        else:
            hasil.append(b)
    return hasil

seksi = []
for nomor in range(2, n_hal):
    baris = rapikan(r.pages[nomor - 1].extract_text() or "")
    i = 0
    while i < len(baris):
        if is_judul(baris[i]):
            j, bag = i, [baris[i]]
            while j + 1 < len(baris) and is_judul(baris[j + 1]):
                j += 1; bag.append(baris[j])
            t = " ".join(bag)
            if seksi and norm(t) == norm(seksi[-1]["judul"]):
                seksi[-1]["hal_akhir"] = nomor
            else:
                seksi.append(dict(judul=t, hal_awal=nomor, hal_akhir=nomor, baris=[]))
            i = j + 1
        else:
            if not seksi:
                seksi.append(dict(judul="(tanpa judul)", hal_awal=nomor, hal_akhir=nomor, baris=[]))
            seksi[-1]["baris"].append(baris[i]); seksi[-1]["hal_akhir"] = nomor
            i += 1

LAB = re.compile(r"^(PENJELASAN\s*:|Persyaratan\s*:?)\s*$")
def kata_baris(ls): return sum(len(l.split()) for l in ls)

def pecah(baris, maks=300, target=250):
    if kata_baris(baris) <= maks:
        return [baris]
    titik = [i for i, l in enumerate(baris) if i > 0 and LAB.match(l)]
    potongan, mulai = [], 0
    for p in titik + [len(baris)]:
        potongan.append(baris[mulai:p]); mulai = p
    gabung = []
    for b in potongan:
        # potongan kecil digabung ke tetangganya (ke belakang atau ke depan)
        if gabung and (kata_baris(b) < 60 or kata_baris(gabung[-1]) < 60):
            gabung[-1] = gabung[-1] + b
        else:
            gabung.append(b)
    akhir = []
    for b in gabung:
        if kata_baris(b) <= maks:
            akhir.append(b); continue
        cur = []
        for l in b:
            if cur and kata_baris(cur) + len(l.split()) > target:
                akhir.append(cur); cur = []
            cur.append(l)
        if cur: akhir.append(cur)
    return akhir

n_duk_sebelum, rincian_pecah = len(seksi), []
for n, sk in enumerate(seksi, start=1):
    bagian_ = pecah(sk["baris"])
    if len(bagian_) > 1:
        rincian_pecah.append((n, sk["judul"], [kata_baris(b) for b in bagian_]))
    assert sum(kata_baris(b) for b in bagian_) == kata_baris(sk["baris"])
    j = sk["judul"]
    sasaran = "OA" if re.search(r"\bOA\b|ORANG ASING", j) else ("WNI" if "WNI" in j else "umum")
    for k, b in enumerate(bagian_):
        teks = "\n".join(b)
        rows.append(dict(
            chunk_id=f"dukcapil_{n:02d}" + (chr(97 + k) if len(bagian_) > 1 else ""),
            sumber="dukcapil", layanan=j,
            bagian=f"bagian {k + 1} dari {len(bagian_)}" if len(bagian_) > 1 else "utuh",
            sumber_url=DUKCAPIL_URL, tanggal_akses=TGL_AKSES_DUKCAPIL,
            tanggal_pembaruan_halaman="tidak tertera",
            teks=teks,
            teks_konteks=f"Layanan kependudukan (Dukcapil): {j}.\n{teks}",
            sumber_ganda="", sasaran=sasaran))

# ================= RINGKASAN =================
df = pd.DataFrame(rows)
df["jumlah_kata"] = df.teks.map(jml_kata)
assert df.chunk_id.is_unique, "chunk_id kembar"
assert (df.teks.str.strip() != "").all(), "ada chunk kosong"
assert not df.teks_konteks.str.contains("imigrasi[A-Z]", regex=True).any(), "nama berkas bocor ke teks_konteks"
kolom = ["chunk_id", "sumber", "layanan", "bagian", "sumber_url", "tanggal_akses",
         "tanggal_pembaruan_halaman", "teks", "teks_konteks", "jumlah_kata", "sumber_ganda", "sasaran"]
df = df[kolom]
df.to_csv(Path(REPO) / "data" / "processed" / "chunks.csv", index=False, encoding="utf-8")

print(f"Imigrasi : {n_imi_sebelum} sub-bagian -> {n_imi_sesudah} chunk (gabung identik)")
print(f"Dukcapil : {n_duk_sebelum} bagian -> {(df.sumber == 'dukcapil').sum()} chunk")
print(f"TOTAL chunk: {len(df)}\n")
print(df.groupby("sumber").jumlah_kata.agg(["count", "min", "median", "max", "sum"]).to_string(), "\n")

k = df.jumlah_kata.tolist()
print(f"< 20 kata (pendek): {sum(x < 20 for x in k)} | < 80 kata: {sum(x < 80 for x in k)} | > 250 kata: {sum(x > 250 for x in k)}")
print("Sasaran:", df.sasaran.value_counts().to_dict())

print("\n--- Bagian Dukcapil yang dipecah ---")
for n, j, ukuran in rincian_pecah:
    print(f"bagian {n}: {j[:60]} -> potongan kata {ukuran}")

print("\n--- Chunk hasil gabungan (identik) ---")
for _, x in df[df.sumber_ganda != ""].iterrows():
    print(f"{x.chunk_id} ({x.jumlah_kata} kata) <- {x.sumber_ganda}")

print("\n--- Nama layanan imigrasi (harus 11, tanpa nama berkas) ---")
nama_unik = sorted({n.strip() for v in df[df.sumber == 'imigrasi'].layanan for n in v.split(';')})
print(len(nama_unik), nama_unik)

print("\n--- Contoh teks_konteks chunk gabungan ---")
print(df[df.sumber_ganda != ""].teks_konteks.iloc[0][:300])

Imigrasi : 55 sub-bagian -> 42 chunk (gabung identik)
Dukcapil : 19 bagian -> 21 chunk
TOTAL chunk: 63

          count  min  median  max   sum
sumber                                 
dukcapil     21   41   116.0  268  2913
imigrasi     42    6    65.0  205  2805 

< 20 kata (pendek): 6 | < 80 kata: 35 | > 250 kata: 1
Sasaran: {'WNI': 51, 'OA': 7, 'umum': 5}

--- Bagian Dukcapil yang dipecah ---
bagian 14: PERPINDAHAN PENDUDUK WNI DALAM NKRI -> potongan kata [250, 89, 191]

--- Chunk hasil gabungan (identik) ---
imi_AnakLahirDiLuarnegri_Persyaratan (12 kata) <- ImigrasiAnakLahirDiLuarnegri; imigrasiPenggantianPasporDiluarNegri
imi_AnakLahirDiLuarnegri_Prosedur (58 kata) <- ImigrasiAnakLahirDiLuarnegri; imigrasiPenggantianPasporDiluarNegri
imi_HajiUmroh_Biaya (28 kata) <- ImigrasiHajiUmroh; imigrasiAnakDwikewarganegaraan; imigrasiAnakdiBawah17; imigrasiCalonPekerjaMigran
imi_PasporRusak_MekanismePenerbitan (64 kata) <- ImigrasiPasporRusak; imigrasiPenggantianPasporDiluarNegri
imi_Paspor

3.5a

In [ ]:
import pandas as pd
from pathlib import Path

P = Path("llm-hallucination/data/processed/chunks.csv")
assert P.exists(), "chunks.csv tidak ada: jalankan ulang sel 3.4 dulu"
df = pd.read_csv(P, dtype=str, keep_default_na=False)
df["jumlah_kata"] = df["jumlah_kata"].astype(int)

kolom = ["chunk_id", "sumber", "layanan", "bagian", "sumber_url", "tanggal_akses",
         "tanggal_pembaruan_halaman", "teks", "teks_konteks", "jumlah_kata",
         "sumber_ganda", "sasaran"]
gagal = []
def cek(nama, ok, info=""):
    print(("LULUS  " if ok else "GAGAL  ") + nama + (f"  [{info}]" if info else ""))
    if not ok: gagal.append(nama)

cek("12 kolom lengkap, urutan benar", list(df.columns) == kolom)
cek("jumlah chunk 63 (42 imigrasi + 21 dukcapil)",
    len(df) == 63 and (df.sumber == "imigrasi").sum() == 42 and (df.sumber == "dukcapil").sum() == 21,
    f"{len(df)} total")
cek("chunk_id unik", df.chunk_id.is_unique)
cek("tidak ada teks kosong", (df.teks.str.strip() != "").all())
cek("jumlah_kata cocok dengan teks", (df.teks.str.split().str.len() == df.jumlah_kata).all())
cek("teks_konteks selalu berakhir dengan teks asli",
    all(k.endswith(t) for k, t in zip(df.teks_konteks, df.teks)))
cek("total kata imigrasi 2805 dan dukcapil 2913",
    df[df.sumber == "imigrasi"].jumlah_kata.sum() == 2805 and df[df.sumber == "dukcapil"].jumlah_kata.sum() == 2913)
cek("tidak ada duplikat (bagian + teks)", not df.duplicated(["bagian", "teks"]).any())
cek("tanggal_pembaruan_halaman terisi semua", (df.tanggal_pembaruan_halaman != "").all())
cek("sasaran hanya WNI/OA/umum", set(df.sasaran) <= {"WNI", "OA", "umum"})
cek("tidak ada sisa penanda halaman web",
    not df.teks.str.contains("Terakhir update|Empty heading", case=False, regex=True).any())
cek("tidak ada nama berkas di teks_konteks", not df.teks_konteks.str.contains("imigrasi[A-Z]", regex=True).any())

print("\n--- Catatan (bukan kegagalan, utang data yang sudah dicatat) ---")
imi = df[df.sumber == "imigrasi"]
print(f"Imigrasi tanpa sumber_url      : {(imi.sumber_url == '').sum()} dari {len(imi)} chunk")
print(f"Imigrasi tanpa tanggal_akses   : {(imi.tanggal_akses == '').sum()} dari {len(imi)} chunk")
print(f"Dukcapil tanpa tanggal_akses   : {((df.sumber == 'dukcapil') & (df.tanggal_akses == '')).sum()}")

print("\nJumlah chunk per sasaran dan sumber:")
print(pd.crosstab(df.sumber, df.sasaran).to_string())
print("\nHASIL AKHIR:", "SEMUA LULUS" if not gagal else f"ADA YANG GAGAL: {gagal}")

LULUS  12 kolom lengkap, urutan benar
LULUS  jumlah chunk 63 (42 imigrasi + 21 dukcapil)  [63 total]
LULUS  chunk_id unik
LULUS  tidak ada teks kosong
LULUS  jumlah_kata cocok dengan teks
LULUS  teks_konteks selalu berakhir dengan teks asli
LULUS  total kata imigrasi 2805 dan dukcapil 2913
LULUS  tidak ada duplikat (bagian + teks)
LULUS  tanggal_pembaruan_halaman terisi semua
LULUS  sasaran hanya WNI/OA/umum
LULUS  tidak ada sisa penanda halaman web
LULUS  tidak ada nama berkas di teks_konteks

--- Catatan (bukan kegagalan, utang data yang sudah dicatat) ---
Imigrasi tanpa sumber_url      : 4 dari 42 chunk
Imigrasi tanpa tanggal_akses   : 42 dari 42 chunk
Dukcapil tanpa tanggal_akses   : 0

Jumlah chunk per sasaran dan sumber:
sasaran   OA  WNI  umum
sumber                 
dukcapil   7    9     5
imigrasi   0   42     0

HASIL AKHIR: SEMUA LULUS


3.5b unduh

In [ ]:
from google.colab import files
files.download("llm-hallucination/data/processed/chunks.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>